# Projeto 8 — Classificação de Ataques de Rede (NSL-KDD) com Random Forest e SVM

Notebook principal para comparação didática entre **Random Forest** e **SVM** com pipeline de pré-processamento, seleção de características, balanceamento e métricas completas.

## 0) Setup Colab (opcional, recomendado no Colab)

Se estiver no Google Colab, execute as células desta seção primeiro para evitar `ModuleNotFoundError: No module named 'src'`.

- Clona o repositório em `/content/Trabalho_IA` quando necessário
- Instala dependências com `requirements.txt`
- Mantém o dataset em `data/raw/` dentro do projeto


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/Trabalho_IA')

if IN_COLAB:
    if not (REPO_DIR / 'src').exists():
        if REPO_DIR.exists():
            import shutil
            shutil.rmtree(REPO_DIR)
        try:
            subprocess.run(['git', 'clone', 'https://github.com/strondinha/Trabalho_IA.git', str(REPO_DIR)], check=True)
        except subprocess.CalledProcessError as exc:
            raise RuntimeError('Falha ao clonar o repositório. Verifique conexão com a internet e tente novamente.') from exc

    os.chdir(REPO_DIR)
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Falha ao instalar dependências de requirements.txt. Verifique o arquivo e tente novamente.') from exc

    print(f'Colab pronto. Diretório atual: {Path.cwd()}')
else:
    print('Ambiente local detectado. Setup Colab ignorado.')


### Dataset no `data/raw/`

Coloque `KDDTrain+.txt` e `KDDTest+.txt` em `data/raw/` (padrão do projeto).

> **Nota:** o loader detecta automaticamente o delimitador (TAB ou vírgula),
> portanto **não é necessário converter os arquivos** — use-os diretamente como
> baixados do site NSL-KDD ou do seu Google Drive.

No Colab, escolha **uma** opção abaixo:
- **Opção A (upload manual)**: envia os arquivos do computador para `data/raw/`
- **Opção B (Google Drive)**: monte o Drive e ajuste `DRIVE_DATA_DIR` (padrão: `/content/drive/MyDrive/nsl-kdd/`)
  para a pasta onde você salvou `KDDTrain+.txt` e `KDDTest+.txt`.


In [ ]:
# Opção A — Upload manual (execute apenas se quiser upload pela sessão atual)
from pathlib import Path
import sys

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

if 'google.colab' in sys.modules:
    from google.colab import files

    uploaded = files.upload()
    for filename in ('KDDTrain+.txt', 'KDDTest+.txt'):
        if filename in uploaded:
            (RAW_DIR / filename).write_bytes(uploaded[filename])

    print('Arquivos encontrados em data/raw/:', sorted(p.name for p in RAW_DIR.glob('KDD*.txt')))
else:
    print('Esta célula é para uso no Colab.')


In [ ]:
# Opção B — Google Drive (execute apenas se quiser usar arquivos persistidos no Drive)
from pathlib import Path
import shutil
import sys

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/nsl-kdd')  # ajuste para sua pasta no Drive

if 'google.colab' in sys.modules:
    from google.colab import drive

    drive.mount('/content/drive')

    missing_files = []
    for filename in ('KDDTrain+.txt', 'KDDTest+.txt'):
        src_file = DRIVE_DATA_DIR / filename
        if src_file.exists():
            shutil.copy2(src_file, RAW_DIR / filename)
        else:
            missing_files.append(str(src_file))

    if missing_files:
        raise FileNotFoundError(
            'Arquivos ausentes no Drive. Garanta que KDDTrain+.txt e KDDTest+.txt estejam em '
            f'{DRIVE_DATA_DIR}/ antes de continuar.\n' + '\n'.join(missing_files)
        )

    print('Arquivos encontrados em data/raw/:', sorted(p.name for p in RAW_DIR.glob('KDD*.txt')))
else:
    print('Esta célula é para uso no Colab.')


## 1) Dataset e organização de pastas

- Dataset oficial (NSL-KDD): https://www.unb.ca/cic/datasets/nsl.html
- Coloque os arquivos abaixo em `data/raw/`:
  - `KDDTrain+.txt`
  - `KDDTest+.txt`

Este notebook lê apenas a partir de `data/raw/`, conforme requisito do projeto.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('/content/Trabalho_IA')]
ROOT = next((p for p in candidates if (p / 'src').exists()), Path.cwd().resolve().parent)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.config import ensure_project_dirs, get_experiment_config
from src.data import load_nsl_kdd_dataset, split_features_target, infer_feature_types
from src.features import stratified_sample_train, get_rf_top_features
from src.models import build_rf_pipeline, build_svm_pipeline, build_experiment_estimators, optional_smote
from src.evaluation import time_fit_and_predict, classification_metrics, plot_confusion, summarize_result

# === Bloco de configuração principal ===
MODE = 'MODO_RAPIDO'      # opções: MODO_RAPIDO | MODO_COMPLETO
SAMPLE_FRAC = None        # None usa o padrão do modo; ou informe valor entre (0,1]
RANDOM_STATE = 42

DIRS = ensure_project_dirs(ROOT)
EXPERIMENT = get_experiment_config(mode=MODE, random_state=RANDOM_STATE, sample_frac=SAMPLE_FRAC)

sns.set_theme(style='whitegrid')
print(f'ROOT detectado: {ROOT}')
print(f"Modo: {EXPERIMENT.mode} | sample_frac={EXPERIMENT.sample_frac} | cv={EXPERIMENT.cv}")
print(f"Pastas: raw={DIRS['raw_dir']} | reports={DIRS['reports_dir']} | figs={DIRS['figures_dir']}")


In [ ]:
raw_dir = DIRS['raw_dir']
train_df, test_df = load_nsl_kdd_dataset(raw_dir=raw_dir)

# Sanity check — mostra shape e o delimitador detectado nos arquivos
for label, df, fname in [('Treino', train_df, 'KDDTrain+.txt'), ('Teste', test_df, 'KDDTest+.txt')]:
    fpath = raw_dir / fname
    with open(fpath, encoding='utf-8', errors='ignore') as _f:
        _line = _f.readline()
    _tabs, _commas = _line.count('\t'), _line.count(',')
    _sep = 'TAB' if _tabs > _commas else 'vírgula'
    print(f'{label}: shape={df.shape}  '
          f'(delimitador detectado: {_sep}; tabs={_tabs}, vírgulas={_commas})')

display(train_df.head())


In [ ]:
X_train, y_train = split_features_target(train_df, target_mode='category')
X_test, y_test = split_features_target(test_df, target_mode='category')

X_train_run, y_train_run = stratified_sample_train(
    X_train,
    y_train,
    sample_frac=EXPERIMENT.sample_frac,
    random_state=EXPERIMENT.random_state,
)

cat_cols, num_cols = infer_feature_types(X_train)
print('Colunas categóricas:', cat_cols)
print('Qtd. colunas numéricas:', len(num_cols))
print(f'Treino original: {X_train.shape} | treino usado ({EXPERIMENT.mode}): {X_train_run.shape}')

print('Distribuição de classes (treino usado):')
display(y_train_run.value_counts())


## 2) Modelos e estratégia experimental

- **Random Forest** e **SVM** com `Pipeline`
- Pré-processamento via `ColumnTransformer`
- Seleção de características: `SelectKBest(mutual_info_classif)` comparada com `passthrough`
- Balanceamento: `class_weight='balanced'`
- Busca de hiperparâmetros: `GridSearchCV`

In [ ]:
rf_pipe = build_rf_pipeline(cat_cols, num_cols, random_state=EXPERIMENT.random_state)
svm_pipe = build_svm_pipeline(cat_cols, num_cols, random_state=EXPERIMENT.random_state)

rf_estimator, svm_estimator = build_experiment_estimators(
    rf_pipe,
    svm_pipe,
    mode=EXPERIMENT.mode,
    cv=EXPERIMENT.cv,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=EXPERIMENT.random_state,
    rf_n_iter=EXPERIMENT.rf_n_iter,
    svm_n_iter=EXPERIMENT.svm_n_iter,
)

print('Estimadores configurados para execução:')
print(f'- RF: {type(rf_estimator).__name__}')
print(f'- SVM: {type(svm_estimator).__name__}')


In [ ]:
# Mantemos o conjunto completo de rótulos para avaliação consistente no teste,
# mesmo quando o treino usa amostragem estratificada (X_train_run/y_train_run).
labels = sorted(pd.unique(pd.concat([y_train, y_test], axis=0)))

rf_pred, rf_train_t, rf_inf_t = time_fit_and_predict(rf_estimator, X_train_run, y_train_run, X_test)
rf_metrics = classification_metrics(y_test, rf_pred, labels=labels)

if hasattr(rf_estimator, 'best_params_'):
    print('Melhores hiperparâmetros RF:')
    print(rf_estimator.best_params_)
print(f'Tempo RF -> treino: {rf_train_t:.2f}s | inferência: {rf_inf_t:.2f}s')
print()
print('Classification report RF:')
print(rf_metrics['classification_report_text'])


In [ ]:
plot_confusion(rf_metrics['confusion_matrix'], labels, title='Random Forest - Matriz de Confusão')
plt.show()

In [ ]:
svm_pred, svm_train_t, svm_inf_t = time_fit_and_predict(svm_estimator, X_train_run, y_train_run, X_test)
svm_metrics = classification_metrics(y_test, svm_pred, labels=labels)

if hasattr(svm_estimator, 'best_params_'):
    print('Melhores hiperparâmetros SVM:')
    print(svm_estimator.best_params_)
else:
    print('SVM em fit único (sem busca) para execução mais rápida.')
print(f'Tempo SVM -> treino: {svm_train_t:.2f}s | inferência: {svm_inf_t:.2f}s')
print()
print('Classification report SVM:')
print(svm_metrics['classification_report_text'])


In [ ]:
plot_confusion(svm_metrics['confusion_matrix'], labels, title='SVM - Matriz de Confusão')
plt.show()

In [ ]:
summary = pd.DataFrame([
    summarize_result('Random Forest', rf_metrics, rf_train_t, rf_inf_t),
    summarize_result('SVM', svm_metrics, svm_train_t, svm_inf_t),
]).sort_values(by='f1_weighted', ascending=False)

top_rf_features = get_rf_top_features(rf_estimator, top_n=15)

summary_path = DIRS['reports_dir'] / 'projeto8_summary.csv'
rf_features_path = DIRS['reports_dir'] / 'projeto8_rf_top_features.csv'
summary.to_csv(summary_path, index=False)
top_rf_features.to_csv(rf_features_path, index=False)

display(summary)
print(f'Resumo salvo em: {summary_path}')

print()
print('Top features do Random Forest (artigo):')
display(top_rf_features)
print(f'Top features salvas em: {rf_features_path}')


## 3) Experimento opcional com SMOTE (fallback automático)

Este passo é opcional. Caso `imbalanced-learn` não esteja instalado, a célula informa e segue com os resultados base (`class_weight`).

In [ ]:
smote = optional_smote(random_state=RANDOM_STATE)
if smote is None:
    print('imbalanced-learn não instalado. Experimento SMOTE ignorado (fallback OK).')
else:
    try:
        from imblearn.pipeline import Pipeline as ImbPipeline
        from sklearn.ensemble import RandomForestClassifier
        from src.models import build_preprocessor

        pre = build_preprocessor(cat_cols, num_cols, scale_numeric=False)
        rf_smote = ImbPipeline(steps=[
            ('preprocessor', pre),
            ('smote', smote),
            ('classifier', RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ])

        y_pred_smote, tr_sm, inf_sm = time_fit_and_predict(rf_smote, X_train, y_train, X_test)
        m_sm = classification_metrics(y_test, y_pred_smote, labels=labels)
        print(m_sm['classification_report_text'])
        print(f'Tempo treino: {tr_sm:.3f}s | inferência: {inf_sm:.3f}s')
    except Exception as exc:
        print(f'Não foi possível executar SMOTE neste ambiente: {exc}')

## 4) Conclusões para apresentação

- Comparar a tabela de métricas (acurácia e F1 ponderado).
- Comparar matriz de confusão por classe (normal, dos, probe, r2l, u2r).
- Discutir trade-off de tempo: RF normalmente treina mais rápido; SVM pode variar conforme kernel e C.
- Destacar que a seleção de características (`SelectKBest`) foi avaliada dentro da busca de hiperparâmetros.